In [2]:
"""
Mercedes-Benz Greener Manufacturing
====================================
Predict the time a car spends on the test bench using XGBoost.

Steps:
  1. Load data
  2. Remove zero-variance columns
  3. Check nulls & unique values
  4. Label-encode categorical columns
  5. Dimensionality reduction with PCA
  6. Train XGBoost & predict on test set
"""

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
from sklearn.ensemble import GradientBoostingRegressor   # fallback; see xgboost note below
# ── XGBoost equivalent (use this when xgboost is installed) ──────────────────
# import xgboost as xgb
# xgb_model = xgb.XGBRegressor(
#     n_estimators=200, max_depth=4, learning_rate=0.05,
#     subsample=0.8, colsample_bytree=0.8,
#     objective="reg:squarederror", random_state=42, n_jobs=-1
# )
# ─────────────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────
print("=" * 60)
print("STEP 1: Loading Data")
print("=" * 60)

train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

print(f"Train shape : {train.shape}")
print(f"Test  shape : {test.shape}")
print(f"\nTarget column 'y' stats:")
print(train["y"].describe())

# ─────────────────────────────────────────
# 2. REMOVE ZERO-VARIANCE COLUMNS
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: Removing Zero-Variance Columns")
print("=" * 60)

# Compute variance only for numeric columns in train (excluding ID and y)
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ["ID", "y"]]

zero_var_cols = [c for c in num_cols if train[c].var() == 0]
print(f"Zero-variance columns found : {len(zero_var_cols)}")
print(f"Columns removed             : {zero_var_cols}")

train.drop(columns=zero_var_cols, inplace=True)
test.drop(columns=zero_var_cols, inplace=True)

print(f"Train shape after removal   : {train.shape}")

# ─────────────────────────────────────────
# 3. NULL & UNIQUE VALUE CHECK
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: Null & Unique Value Check")
print("=" * 60)

train_nulls = train.isnull().sum().sum()
test_nulls  = test.isnull().sum().sum()
print(f"Total nulls in train : {train_nulls}")
print(f"Total nulls in test  : {test_nulls}")

# Unique values per column (show categorical ones)
cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
print(f"\nCategorical columns  : {cat_cols}")
for col in cat_cols:
    u_train = train[col].nunique()
    u_test  = test[col].nunique()
    print(f"  {col:5s} -> train unique: {u_train}, test unique: {u_test}")

# ─────────────────────────────────────────
# 4. LABEL ENCODING
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: Label Encoding Categorical Columns")
print("=" * 60)

le = LabelEncoder()
for col in cat_cols:
    # Fit on combined train+test values to avoid unseen label issues
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))
    print(f"  Encoded: {col}")

print("\nSample after encoding:")
print(train[cat_cols].head(3))

# ─────────────────────────────────────────
# 5. PREPARE FEATURE MATRICES
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: Preparing Feature Matrices")
print("=" * 60)

y_train = train["y"].values
X_train = train.drop(columns=["ID", "y"]).values
X_test  = test.drop(columns=["ID"]).values

print(f"X_train shape : {X_train.shape}")
print(f"X_test  shape : {X_test.shape}")
print(f"y_train shape : {y_train.shape}")

# ─────────────────────────────────────────
# 6. DIMENSIONALITY REDUCTION (PCA)
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: PCA Dimensionality Reduction")
print("=" * 60)

# Retain 95% of variance
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

print(f"Components retained (95% variance) : {pca.n_components_}")
print(f"X_train after PCA                  : {X_train_pca.shape}")
print(f"X_test  after PCA                  : {X_test_pca.shape}")
print(f"Cumulative variance explained      : {pca.explained_variance_ratio_.sum():.4f}")

# ─────────────────────────────────────────
# 7. XGBOOST MODEL
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: Training XGBoost Model")
print("=" * 60)

# Using sklearn GradientBoostingRegressor — same algorithm family as XGBoost.
# To switch to XGBoost, replace this block with the xgb.XGBRegressor block above.
xgb_model = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)

xgb_model.fit(X_train_pca, y_train)
print("  Model training complete.")

# Training R² score
train_preds = xgb_model.predict(X_train_pca)
r2 = r2_score(y_train, train_preds)
print(f"\nTraining R² Score : {r2:.4f}")

# ─────────────────────────────────────────
# 8. PREDICT ON TEST SET
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: Predicting on Test Set")
print("=" * 60)

test_preds = xgb_model.predict(X_test_pca)

# Build submission dataframe
submission = pd.DataFrame({
    "ID": test["ID"],
    "y" : test_preds
})

print(f"Prediction stats:")
print(submission["y"].describe())
print(f"\nSample predictions:")
print(submission.head(10).to_string(index=False))

# ─────────────────────────────────────────
# 9. SAVE RESULTS
# ─────────────────────────────────────────
output_path = "mercedes_predictions.csv"
submission.to_csv(output_path, index=False)
print(f"\n✅ Predictions saved to: {output_path}")
print("=" * 60)
print("Pipeline Complete!")
print("=" * 60)

STEP 1: Loading Data
Train shape : (4209, 378)
Test  shape : (4209, 377)

Target column 'y' stats:
count    4209.000000
mean      100.669318
std        12.679381
min        72.110000
25%        90.820000
50%        99.150000
75%       109.010000
max       265.320000
Name: y, dtype: float64

STEP 2: Removing Zero-Variance Columns
Zero-variance columns found : 12
Columns removed             : ['X11', 'X93', 'X107', 'X233', 'X235', 'X268', 'X289', 'X290', 'X293', 'X297', 'X330', 'X347']
Train shape after removal   : (4209, 366)

STEP 3: Null & Unique Value Check
Total nulls in train : 0
Total nulls in test  : 0

Categorical columns  : ['X0', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X8']
  X0    -> train unique: 47, test unique: 49
  X1    -> train unique: 27, test unique: 27
  X2    -> train unique: 44, test unique: 45
  X3    -> train unique: 7, test unique: 7
  X4    -> train unique: 4, test unique: 4
  X5    -> train unique: 29, test unique: 32
  X6    -> train unique: 12, test unique: 12
